# Build and test a pricing policy
Copy this notebook, write your policy in section 4, and run all. Every policy faces the same customers, so only prices differ.

## 1. Setup
Works locally and in Colab.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not Path("ieor145").exists():
        subprocess.run(["git", "clone", "https://github.com/zhouhaitz/ieor145"], check=True)
    os.chdir("ieor145")
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

sys.path.insert(0, str(Path.cwd()))
print("Repo root:", Path.cwd())

## 2. Load the bakery

In [ ]:
from bakery_sim.config import load_scenario

scenario = load_scenario("configs/baseline.json")
print(scenario.open_hours, scenario.price_ladder)

## 3. What a policy is
An object with a `name` and a `prices(context)` method, called once per open hour.
`context` has `day_index`, `day_of_week`, `hour`, `inventory` (units per `(product_id, age)`; age 1 = day-old) and `scenario`.
Return a `PriceDecision` for every item with units > 0, at a level on `scenario.price_ladder`.

Useful building blocks: expected customers of each type per hour, and MNL purchase probabilities.

In [ ]:
from bakery_sim.arrivals import expected_arrivals_table
from bakery_sim.choice import choice_probabilities

arrivals = expected_arrivals_table(scenario)
print(arrivals[arrivals["day_of_week"] == "Mon"].pivot(index="hour", columns="type_id", values="expected_arrivals").round(1))

regular = scenario.customer_type("regular")
shelf = [("croissant", 0), ("sourdough", 0), ("sourdough", 1)]
example_prices = {("croissant", 0): 5.00, ("sourdough", 0): 7.00, ("sourdough", 1): 3.85}
choice_probabilities(scenario, regular, shelf, example_prices)  # None = leaves without buying

## 4. Your policy
Replace the body of `prices`. The example below always charges menu price; `bakery_sim/example_policies.py` has a full MNL example.

In [ ]:
from bakery_sim.interfaces import PriceDecision
from bakery_sim.pricing import ladder_price


class MyPolicy:
    """Describe your policy in one line."""

    name = "my_policy"

    def prices(self, context):
        decisions = {}
        for (product_id, age), units in context.inventory.items():
            if units == 0:
                continue
            level = 1.0 if age == 0 else 0.55  # TODO: your rule (e.g. LP bid prices) goes here
            price = ladder_price(context.scenario, product_id, context.day_index, level).price
            decisions[(product_id, age)] = PriceDecision(price=price, level=level, reason="policy")
        return decisions

## 5. Compare against the baselines
Same 30 test days for every policy. Numbers are per day.

In [ ]:
from bakery_sim.evaluate import compare_policies
from bakery_sim.example_policies import MyopicMnlPolicy
from bakery_sim.pricing import FixedPricePolicy, HistoricalPricingPolicy

policies = [FixedPricePolicy(), HistoricalPricingPolicy(scenario), MyopicMnlPolicy(), MyPolicy()]
results = compare_policies(scenario, policies)
results.round(2)

In [ ]:
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(6.5, 4))
axis.scatter(results["units_wasted_per_day"], results["profit_per_day"], color="#3182bd")
labels = results.groupby(["units_wasted_per_day", "profit_per_day"])["policy"].apply(" = ".join)  # identical results share a label
for (units_wasted, profit), label in labels.items():
    axis.annotate(label, (units_wasted, profit), textcoords="offset points", xytext=(6, 4))
axis.margins(x=0.25, y=0.15)
axis.set_xlabel("Units wasted per day")
axis.set_ylabel("Profit per day ($)")
axis.set_title("Profit vs. waste by policy (up and left is better)")
plt.tight_layout()
plt.show()

## 6. Try another world (optional)
Re-run section 5 after changing a knob, e.g. bake more or make leftovers worthless.

In [ ]:
# scenario = scenario.with_overrides(supply_ratio=1.4)      # more leftovers
# scenario = scenario.with_overrides(salvage_fraction=0.0)  # leftovers thrown away